**Week 1: Project Initialization and Dataset Acquisition.**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive')

base_dir = '/content/drive/MyDrive/VOC2012_train_val/VOC2012_train_val'
image_dir = os.path.join(base_dir, 'JPEGImages')
mask_dir = os.path.join(base_dir, 'SegmentationClass')

all_image_names = sorted(os.listdir(image_dir))
images_with_any_object = []

object_class_ids = list(range(1, 21))

count = 0
for image_name in all_image_names:
    mask_name = image_name.replace('.jpg', '.png')
    mask_path = os.path.join(mask_dir, mask_name)

    if os.path.exists(mask_path):
        try:
            mask = Image.open(mask_path)
            mask_np = np.array(mask)


            if any(obj_id in np.unique(mask_np) for obj_id in object_class_ids):
                images_with_any_object.append(image_name)
                count += 1
                if count >= 10:
                    break
        except Exception:
            continue


for image_name in images_with_any_object:
    image_path = os.path.join(image_dir, image_name)
    mask_name = image_name.replace('.jpg', '.png')
    mask_path = os.path.join(mask_dir, mask_name)

    try:
        original_image = Image.open(image_path).convert("RGB")
        original_mask = Image.open(mask_path)

        mask_np = np.array(original_mask)

        all_objects_mask = (mask_np != 0)

        segmented_image_np = np.zeros_like(np.array(original_image))
        segmented_image_np[all_objects_mask] = np.array(original_image)[all_objects_mask]

        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        fig.suptitle(f'Segmented Image: {image_name}', fontsize=16)

        ax.imshow(segmented_image_np.astype(np.uint8))
        ax.set_title('Segmented Image')
        ax.axis('off')

        plt.show()

    except Exception as e:
        print(f"Failed to process {image_name}: {e}")

**Week 2: Data Preprocessing and Validation.**

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image


target_size = (256, 256)
resize_transform = T.Resize(target_size, interpolation=Image.NEAREST)
normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

class VOCSegmentationDataset(Dataset):
    def __init__(self, filenames, image_dir, mask_dir, resize_transform, normalize_transform, object_class_ids):
        self.filenames = filenames
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.resize_transform = resize_transform
        self.normalize_transform = normalize_transform
        self.object_class_ids = object_class_ids

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        image_name = self.filenames[idx]
        image_path = os.path.join(self.image_dir, image_name)
        mask_name = image_name.replace('.jpg', '.png')
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path)

        image = self.resize_transform(image)
        mask = self.resize_transform(mask)

        if random.random() > 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        rotation_degrees = random.randint(-10, 10)
        image = TF.rotate(image, rotation_degrees)
        mask = TF.rotate(mask, rotation_degrees)

        mask_np = np.array(mask)
        binary_mask = np.zeros_like(mask_np, dtype=np.float32)

        for obj_id in self.object_class_ids:
            binary_mask[mask_np == obj_id] = 1

        image = T.ToTensor()(image)
        image = self.normalize_transform(image)
        binary_mask = torch.from_numpy(binary_mask).unsqueeze(0)

        return image, binary_mask

random.seed(42)
random.shuffle(images_with_any_object)
train_split = int(len(images_with_any_object) * 0.8)
val_split = int(len(images_with_any_object) * 0.1)

train_filenames = images_with_any_object[:train_split]
val_filenames = images_with_any_object[train_split:train_split + val_split]
test_filenames = images_with_any_object[train_split + val_split:]

train_dataset = VOCSegmentationDataset(train_filenames, image_dir, mask_dir, resize_transform, normalize, object_class_ids)
val_dataset = VOCSegmentationDataset(val_filenames, image_dir, mask_dir, resize_transform, normalize, object_class_ids)
test_dataset = VOCSegmentationDataset(test_filenames, image_dir, mask_dir, resize_transform, normalize, object_class_ids)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

object_class_ids = list(range(1, 21))

if not images_with_any_object:
    print("No images with objects were found to visualize.")
else:
    images_to_display = random.sample(images_with_any_object, min(3, len(images_with_any_object)))

    for image_name in images_to_display:
        image_path = os.path.join(image_dir, image_name)
        mask_name = image_name.replace('.jpg', '.png')
        mask_path = os.path.join(mask_dir, mask_name)

        try:
            original_image = Image.open(image_path).convert("RGB")
            original_mask = Image.open(mask_path)

            mask_np = np.array(original_mask)
            all_objects_mask = (mask_np != 0)

            segmented_image_np = np.zeros_like(np.array(original_image))
            segmented_image_np[all_objects_mask] = np.array(original_image)[all_objects_mask]

            segmented_image_pil = Image.fromarray(segmented_image_np.astype(np.uint8))

            fig, axes = plt.subplots(1, 4, figsize=(15, 5))
            fig.suptitle(f'Rotation of Segmented Image: {image_name}', fontsize=16)

            axes[0].imshow(segmented_image_pil)
            axes[0].set_title('Original Segmented')
            axes[0].axis('off')

            for i in range(3):
                rotation_degrees = random.randint(-45, 45)
                rotated_image = F.rotate(segmented_image_pil, rotation_degrees)

                axes[i+1].imshow(rotated_image)
                axes[i+1].set_title(f'Rotated by {rotation_degrees}°')
                axes[i+1].axis('off')

            plt.show()

        except Exception as e:
            print(f"Failed to process {image_name}: {e}")



In [ ]:
all_image_names = sorted(os.listdir(image_dir))
images_with_bench = []
bench_class_id = 10

count = 0
for image_name in all_image_names:
    mask_name = image_name.replace('.jpg', '.png')
    mask_path = os.path.join(mask_dir, mask_name)

    if os.path.exists(mask_path):
        try:
            mask = Image.open(mask_path)
            mask_np = np.array(mask)

            if bench_class_id in np.unique(mask_np):
                images_with_bench.append(image_name)
                count += 1
                if count >= 10:
                    break
        except Exception:
            continue

print(f"Found {len(images_with_bench)} images that contain a bench.")

if not images_with_bench:
    print("No images with benches found. Please run the filtering step first.")
else:
    images_to_display = random.sample(images_with_bench, min(5, len(images_with_bench)))

    for image_name in images_to_display:
        image_path = os.path.join(image_dir, image_name)
        mask_name = image_name.replace('.jpg', '.png')
        mask_path = os.path.join(mask_dir, mask_name)

        try:
            original_image = Image.open(image_path).convert("RGB")
            original_mask = Image.open(mask_path)

            mask_np = np.array(original_mask)

            bench_binary_mask = (mask_np == bench_class_id).astype(np.uint8)

            segmented_bench_np = np.zeros_like(np.array(original_image))
            segmented_bench_np[bench_binary_mask == 1] = np.array(original_image)[bench_binary_mask == 1]

            fig, ax = plt.subplots(1, 2, figsize=(12, 6))
            fig.suptitle(f'Bench Segmentation: {image_name}', fontsize=16)

            ax[0].imshow(original_image)
            ax[0].set_title('Original Image')
            ax[0].axis('off')

            ax[1].imshow(segmented_bench_np)
            ax[1].set_title('Binary Segmented (Bench only)')
            ax[1].axis('off')

            plt.show()

        except Exception as e:
            print(f"Failed to process {image_name}: {e}")



**Week 3: Initial Model Training.**

In [ ]:
model = models.deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("✅ Pre-trained DeepLabV3 model loaded.")

import torch
import torch.nn as nn
import torch.optim as optim
import time

def train_and_validate(model, train_loader, val_loader, num_epochs, device):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-5) # Changed learning rate

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        start_time = time.time()
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)["out"]
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)["out"]
                loss = criterion(outputs, masks)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        end_time = time.time()
        duration = end_time - start_time

        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Time: {duration:.2f}s")
    return model

trained_model = train_and_validate(model, train_loader, val_loader, num_epochs=5, device=device)

torch.save(trained_model.state_dict(), "my_deeplabv3_model.pth")
print("✅ Model saved as my_deeplabv3_model.pth")

import torch
import numpy as np

def iou_score(outputs, masks):
    predicted_masks = torch.sigmoid(outputs)
    predicted_masks = (predicted_masks > 0.5).float()
    outputs_flat = predicted_masks.view(-1)
    masks_flat = masks.view(-1)
    intersection = (outputs_flat * masks_flat).sum()
    union = outputs_flat.sum() + masks_flat.sum() - intersection
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.item()

def dice_score(outputs, masks):
    predicted_masks = torch.sigmoid(outputs)
    predicted_masks = (predicted_masks > 0.5).float()
    outputs_flat = predicted_masks.view(-1)
    masks_flat = masks.view(-1)
    intersection = (outputs_flat * masks_flat).sum()
    dice = (2. * intersection + 1e-6) / (outputs_flat.sum() + masks_flat.sum() + 1e-6)
    return dice.item()

def pixel_accuracy(outputs, masks):
    predicted_masks = torch.sigmoid(outputs)
    predicted_masks = (predicted_masks > 0.5).float()
    correct_pixels = (predicted_masks == masks).float().sum()
    total_pixels = torch.numel(masks)
    accuracy = correct_pixels / total_pixels
    return accuracy.item()

metrics = {"IoU": [], "Dice": [], "PixelAcc": []}
trained_model.eval()

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = trained_model(imgs)["out"]
        metrics["IoU"].append(iou_score(outputs, masks))
        metrics["Dice"].append(dice_score(outputs, masks))
        metrics["PixelAcc"].append(pixel_accuracy(outputs, masks))

print(f"Mean IoU: {np.mean(metrics['IoU']):.4f}")
print(f"Mean Dice: {np.mean(metrics['Dice']):.4f}")
print(f"Pixel Accuracy: {np.mean(metrics['PixelAcc']):.4f}")



**Week 4: Predictions and Fine-tuning.**

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models.segmentation as models
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Define Hyperparameters and Model Configuration
num_epochs = 30
batch_size = 16
patience = 3
num_classes = 2 # Background + Object
image_size = 256
weight_decay = 1e-5

def train_model(model, train_loader, val_loader, lr, num_epochs=num_epochs, patience=patience):
    best_val_loss = float("inf")
    patience_counter = 0
    backbone_frozen = True
    best_model_wts = None

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    weight_tensor = torch.tensor([1.5], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=weight_tensor)

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} [Training]", leave=False):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)["out"]
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_train_loss = total_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{num_epochs} [Validation]", leave=False):
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)["out"]
                loss = criterion(outputs, masks)
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch}/{num_epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_wts = model.state_dict()
            torch.save(best_model_wts, f"deeplabv3_best_lr_{lr}.pth")
            print("Best model updated!")
            patience_counter = 0
        else:
            patience_counter += 1

        if backbone_frozen and epoch >= 2:
            for param in model.backbone.parameters():
                param.requires_grad = True
            print("Backbone unfrozen, fine-tuning full model!")
            backbone_frozen = False

        scheduler.step()
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

    if best_model_wts:
        model.load_state_dict(best_model_wts)
    return model




**Week 5: Improve Data and Experiment with Architectures**

In [ ]:
best_lr = 5e-5

print(f"\n--- Starting Final Training with Best Learning Rate: {best_lr} ---")

model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
model.to(device)

trained_model = train_model(model, train_loader, val_loader, lr=best_lr)

print("\n--- Final Training Complete ---")

evaluate_model(trained_model, val_loader)

visualize_predictions(trained_model, val_loader, num_samples=8)

**Week 6: Inference**

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
import numpy as np
import torchvision.transforms as T
import torchvision.models.segmentation as models
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
import matplotlib.pyplot as plt
import os
import random
from google.colab import files

files.download('deeplabv3_best_lr_5e-05.pth')
# --- 1. Setup ---
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive')

# --- 2. Load the model architecture and weights ---
# Instantiate the DeepLabV3 architecture (must be the same as in training)
model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)

# Set the path to your saved model file in Google Drive
model_path = '/content/drive/MyDrive/deeplabv3_best_lr_5e-05.pth' # <-- Check this path is correct
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)

# Set the model to evaluation mode
model.eval()

# --- 3. Load and preprocess the new images ---
# Set the path to the folder containing your images
image_folder_path = '/content/drive/MyDrive/random_images' # <-- This is your folder's path
image_files = [f for f in os.listdir(image_folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))]

if not image_files:
    print("No images found in the specified folder.")
else:
    # Preprocessing transforms (must match your training transforms)
    resize_transform = T.Resize((256, 256), interpolation=Image.NEAREST)
    normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    # --- 4. Make predictions and visualize ---
    # Set to the number of images you have (16)
    num_samples = len(image_files)

    for image_name in image_files:
        image_path = os.path.join(image_folder_path, image_name)
        image_pil = Image.open(image_path).convert("RGB")

        # Apply transforms
        image_tensor = T.ToTensor()(resize_transform(image_pil))
        image_tensor = normalize(image_tensor).unsqueeze(0) # Add a batch dimension

        # Make a prediction
        with torch.no_grad():
            outputs = model(image_tensor.to(device))['out']
            predicted_mask = (torch.sigmoid(outputs) > 0.5).float().squeeze(0).cpu().numpy().squeeze()

        # Undo normalization to get the original image colors for visualization
        un_normalize = T.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225], std=[1/0.229, 1/0.224, 1/0.225])
        original_image = un_normalize(image_tensor.squeeze(0)).cpu().permute(1, 2, 0).numpy()
        original_image = np.clip(original_image, 0, 1)

        # Create a new image with a black background and the object in its original color
        masked_image_pred = np.zeros_like(original_image)
        masked_image_pred[predicted_mask.astype(bool)] = original_image[predicted_mask.astype(bool)]

        # Visualize the prediction with 3 subplots
        fig, axs = plt.subplots(1, 3, figsize=(18, 6))

        # 1. Original Image
        axs[0].imshow(original_image); axs[0].set_title(f'Original Image: {image_name}'); axs[0].axis('off')

        # 2. Predicted Binary Mask (Pure black and white output)
        axs[1].imshow(predicted_mask, cmap='gray', vmin=0, vmax=1); axs[1].set_title('Predicted Binary Mask'); axs[1].axis('off')

        # 3. Predicted Masked Image (Color on black background)
        axs[2].imshow(masked_image_pred); axs[2].set_title('Predicted Masked Image'); axs[2].axis('off')

        plt.show()
